In [ ]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [2]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("/Users/arjolaarapi-gjini/Desktop/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0
...,...,...
995,So what's the latest? It sounds contradictory ...,0
996,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",1
997,Barb I will call to explain. Are you back in t...,0
998,Yang on travelNot free tonite.May work tomorrow,0


### Let's divide the training and test set into two partitions

In [6]:
# Train validitation split
from sklearn.model_selection import train_test_split

# Split original training data into train + validation (80% / 20%)
data_train, data_val = train_test_split(
    data, 
    test_size=0.2,           # 20% goes to validation
    random_state=42,         # reproducible split
    stratify=data['label']   # preserves SPAM/HAM ratio
)

# Check the resulting shapes
print("Training set shape:", data_train.shape)
print("Validation set shape:", data_val.shape)

Training set shape: (800, 2)
Validation set shape: (200, 2)


## Data Preprocessing

In [18]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [19]:
# Your code
import re

# Preprocessing function: clean HTML + remove special chars/numbers + lowercase + stem/lemmatize
def preprocess_text(text):
    # 1. Remove inline JavaScript/CSS
    text = re.sub(r'<script.*?>.*?</script>', ' ', text, flags=re.DOTALL)
    text = re.sub(r'<style.*?>.*?</style>', ' ', text, flags=re.DOTALL)
    
    # 2. Remove HTML comments
    text = re.sub(r'<!--.*?-->', ' ', text, flags=re.DOTALL)
    
    # 3. Remove remaining HTML tags
    text = re.sub(r'<.*?>', ' ', text)
    
    # 4. Remove numbers
    text = re.sub(r'\d+', ' ', text)
    
    # 5. Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # 6. Remove single characters
    text = re.sub(r'\b[a-zA-Z]\b', ' ', text)
    
    # 7. Remove multiple spaces
    text = re.sub(r'\s+', ' ', text)
    
    # 8. Convert to lowercase
    text = text.lower()
    
    # 9. Remove stopwords
    words = [w for w in text.split() if w not in stopwords.words('english')]
    
    # 10. Stem using SnowballStemmer
    words = [snowball.stem(w) for w in words]
    
    return " ".join(words)

# Apply to all partitions
data_train['preprocessed_text'] = data_train['text'].apply(preprocess_text)
data_val['preprocessed_text'] = data_val['text'].apply(preprocess_text)
data_test['preprocessed_text'] = data_test['text'].apply(preprocess_text)

NameError: name 'data_test' is not defined

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [ ]:
# Your code
import re
import string

def clean_text(text):
    # 1. Remove all special characters
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # 2. Remove numbers
    text = re.sub(r'\d+', ' ', text)
    
    # 3. Remove all single characters
    text = re.sub(r'\b[a-zA-Z]\b', ' ', text)
    
    # 4. Remove single characters from the start
    text = re.sub(r'^[a-zA-Z]\s+', '', text)
    
    # 5. Substitute multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)
    
    # 6. Remove prefixed 'b' (common in byte strings)
    text = re.sub(r'\bb\b', '', text)
    
    # 7. Convert to lowercase
    text = text.lower()
    
    return text.strip()

# Apply to all partitions
data_train['preprocessed_text'] = data_train['preprocessed_text'].apply(clean_text)
data_val['preprocessed_text'] = data_val['preprocessed_text'].apply(clean_text)
data_test['preprocessed_text'] = data_test['preprocessed_text'].apply(clean_text)

## Now let's work on removing stopwords
Remove the stopwords.

In [ ]:
# Your code
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    # Split text into words, remove stopwords, join back
    words = [word for word in text.split() if word not in stop_words]
    return " ".join(words)

# Apply to all partitions
data_train['preprocessed_text'] = data_train['preprocessed_text'].apply(remove_stopwords)
data_val['preprocessed_text'] = data_val['preprocessed_text'].apply(remove_stopwords)
data_test['preprocessed_text'] = data_test['preprocessed_text'].apply(remove_stopwords)

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [ ]:
# Your code
# Function to stem / lemmatize text
def stem_text(text):
    words = text.split()  # split into words
    stemmed_words = [snowball.stem(word) for word in words]  # stem each word
    return " ".join(stemmed_words)

# Apply to all partitions
data_train['preprocessed_text'] = data_train['preprocessed_text'].apply(stem_text)
data_val['preprocessed_text'] = data_val['preprocessed_text'].apply(stem_text)
data_test['preprocessed_text'] = data_test['preprocessed_text'].apply(stem_text)

## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [10]:
# Your code
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Initialize CountVectorizer
vectorizer = CountVectorizer(max_features=1000)  # limit features for speed
X_train_counts = vectorizer.fit_transform(data_train['preprocessed_text'])

# Create a dataframe of words and their counts
word_counts = pd.DataFrame(X_train_counts.toarray(), columns=vectorizer.get_feature_names_out())
word_counts['label'] = data_train['label'].values

# Sum words separately for ham and spam
ham_words = word_counts[word_counts['label']=='ham'].drop(columns='label').sum().sort_values(ascending=False).head(10)
spam_words = word_counts[word_counts['label']=='spam'].drop(columns='label').sum().sort_values(ascending=False).head(10)

print("Top 10 words in HAM messages:")
print(ham_words)
print("\nTop 10 words in SPAM messages:")
print(spam_words)

Top 10 words in HAM messages:
abacha       0
pm           0
permit       0
person       0
peter        0
petroleum    0
philippin    0
phone        0
place        0
plan         0
dtype: int64

Top 10 words in SPAM messages:
abacha       0
pm           0
permit       0
person       0
peter        0
petroleum    0
philippin    0
phone        0
place        0
plan         0
dtype: int64


## Extra features

In [11]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

,text,label,preprocessed_text,money_mark,suspicious_words,text_len
442,Dear=2C Good day hope fine=2Cdear am writting ...,1,dear good day hope fine cdear writ mail due re...,1,1,902
962,FROM MR HENRY KABORETHE CHIEF AUDITOR INCHARGE...,1,mr henri kaboreth chief auditor inchargeforeig...,0,1,1735
971,Will do.,0,,0,0,0
190,FROM THE DESK OF DR.ADAMU ISMALERAUDITING AND...,1,desk dradamu ismaleraudit account managerbank ...,1,1,355
551,"Dear Friend, My name is LOI C.ESTRADA,The wife...",1,dear friend name loi cestradath wife mr joseph...,1,1,1347


## How would work the Bag of Words with Count Vectorizer concept?

In [12]:
# Your code
from sklearn.feature_extraction.text import CountVectorizer

texts = ["I love pizza", "Pizza is love"]
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts)

print(vectorizer.get_feature_names_out())  # ['is', 'love', 'pizza']
print(X.toarray())  
# Output:
# [[0 1 1]
#  [1 1 1]]

['is' 'love' 'pizza']
[[0 1 1]
 [1 1 1]]


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [14]:
# Your code
from sklearn.feature_extraction.text import TfidfVectorizer

# Load the vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=1000)  # limit features for speed

# Vectorize training and validation sets
X_train_tfidf = tfidf_vectorizer.fit_transform(data_train['preprocessed_text'])
X_val_tfidf = tfidf_vectorizer.transform(data_val['preprocessed_text'])

# Print the shape
print("TF-IDF shape of training set:", X_train_tfidf.shape)
print("TF-IDF shape of validation set:", X_val_tfidf.shape)

TF-IDF shape of training set: (800, 1000)
TF-IDF shape of validation set: (200, 1000)


## And the Train a Classifier?

In [15]:
# Your code
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Initialize the model
model = MultinomialNB()

# 2. Train the model on TF-IDF features
model.fit(X_train_tfidf, data_train['label'])

# 3. Predict on validation set
val_predictions = model.predict(X_val_tfidf)

# 4. Evaluate
accuracy = accuracy_score(data_val['label'], val_predictions)
print("Validation Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(data_val['label'], val_predictions))
print("\nConfusion Matrix:\n", confusion_matrix(data_val['label'], val_predictions))

Validation Accuracy: 0.96

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.95      0.96       112
           1       0.93      0.98      0.96        88

    accuracy                           0.96       200
   macro avg       0.96      0.96      0.96       200
weighted avg       0.96      0.96      0.96       200


Confusion Matrix:
 [[106   6]
 [  2  86]]


### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [16]:
# Your code
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import numpy as np

# -----------------------------
# 1. Prepare Text‑Only Feature Matrices
# -----------------------------

# Bag‑of‑Words (BoW)
count_vect = CountVectorizer(max_features=2000)
X_train_bow = count_vect.fit_transform(data_train['preprocessed_text'])
X_val_bow = count_vect.transform(data_val['preprocessed_text'])

# TF‑IDF
tfidf_vect = TfidfVectorizer(max_features=2000)
X_train_tfidf = tfidf_vect.fit_transform(data_train['preprocessed_text'])
X_val_tfidf = tfidf_vect.transform(data_val['preprocessed_text'])

# -----------------------------
# 2. Extra Numeric Features
# -----------------------------

extra_features_train = data_train[['money_mark','suspicious_words','text_len']].values
extra_features_val = data_val[['money_mark','suspicious_words','text_len']].values

# -----------------------------
# 3. Combine Features (if needed)
# -----------------------------

# BoW + extra
X_train_bow_extra = np.hstack([X_train_bow.toarray(), extra_features_train])
X_val_bow_extra = np.hstack([X_val_bow.toarray(), extra_features_val])

# TF‑IDF + extra
X_train_tfidf_extra = np.hstack([X_train_tfidf.toarray(), extra_features_train])
X_val_tfidf_extra = np.hstack([X_val_tfidf.toarray(), extra_features_val])

# -----------------------------
# 4. Train and Evaluate MultinomialNB
# -----------------------------

def train_and_eval(X_train, X_val, desc):
    model = MultinomialNB()                # default parameters
    model.fit(X_train, data_train['label'])
    preds = model.predict(X_val)
    
    acc = accuracy_score(data_val['label'], preds)
    print(f"--- {desc} ---")
    print("Validation Accuracy:", acc)
    print(classification_report(data_val['label'], preds))
    print()

# Evaluate each setup
train_and_eval(X_train_bow, X_val_bow, "Bag of Words only")
train_and_eval(X_train_tfidf, X_val_tfidf, "TF‑IDF only")
train_and_eval(X_train_bow_extra, X_val_bow_extra, "BoW + Extra Flags")
train_and_eval(X_train_tfidf_extra, X_val_tfidf_extra, "TF‑IDF + Extra Flags")

--- Bag of Words only ---
Validation Accuracy: 0.955
              precision    recall  f1-score   support

           0       0.98      0.94      0.96       112
           1       0.92      0.98      0.95        88

    accuracy                           0.95       200
   macro avg       0.95      0.96      0.95       200
weighted avg       0.96      0.95      0.96       200


--- TF‑IDF only ---
Validation Accuracy: 0.98
              precision    recall  f1-score   support

           0       0.99      0.97      0.98       112
           1       0.97      0.99      0.98        88

    accuracy                           0.98       200
   macro avg       0.98      0.98      0.98       200
weighted avg       0.98      0.98      0.98       200


--- BoW + Extra Flags ---
Validation Accuracy: 0.95
              precision    recall  f1-score   support

           0       1.00      0.91      0.95       112
           1       0.90      1.00      0.95        88

    accuracy                 